### Task 3: Quantized MedSurg stay-days prediction (<=2 vs >2)

In [1]:
# Clean baseline version WITHOUT label leakage
# ------------------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

# ------------------------------------------------------------
# 0) Configuration
# ------------------------------------------------------------
DATA_PATH = Path("/Users/kolwu/Downloads/patient_features_new.csv")   # change to your local path if needed
SAVE_DIR  = Path("/Users/kolwu/Downloads")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

FILTER_TO_MEDSURG = False  # set True if you only want MedSurg encounters

In [2]:
# ------------------------------------------------------------
# 1) Load data
# ------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

In [3]:
# ------------------------------------------------------------
# 2) Build the target (label) — quantized MedSurg stay days
# ------------------------------------------------------------
if "encounter_duration_days" not in df.columns:
    # fallback: compute from timestamps if duration column missing
    start_col = next((c for c in df.columns if "start" in c and "dt" in c), None)
    end_col   = next((c for c in df.columns if "end" in c and "dt" in c), None)
    if not (start_col and end_col):
        raise ValueError("No stay duration columns found to create the target.")
    dt_start  = pd.to_datetime(df[start_col], errors="coerce")
    dt_end    = pd.to_datetime(df[end_col], errors="coerce")
    stay_days = (dt_end - dt_start).dt.total_seconds() / (60*60*24)
else:
    stay_days = pd.to_numeric(df["encounter_duration_days"], errors="coerce")

df = df[np.isfinite(stay_days)].copy()
df["target_quantized"] = (stay_days > 2).astype(int)  # 0 = low (<=2 days), 1 = high (>2 days)

In [4]:
# ------------------------------------------------------------
# 3) (Optional) Keep only MedSurg encounters
# ------------------------------------------------------------
service_col = None
for cand in ["encounter_medical_service", "MEDICAL_SERVICE", "medical_service"]:
    if cand in df.columns:
        service_col = cand
        break

if FILTER_TO_MEDSURG and service_col is not None:
    mask_medsurg = df[service_col].astype(str).str.lower().str.contains(r"med[\s\-_/]*surg|medsurg", na=False)
    df = df[mask_medsurg].copy()

In [5]:
# ------------------------------------------------------------
# 4) Feature engineering (no leakage)
# ------------------------------------------------------------
def is_leaky(col: str) -> bool:
    """Exclude any feature that leaks outcome information."""
    c = col.lower()
    bad_tokens = ["duration", "end_dt", "discharge", "death_dt", "expired", "deceased_dt"]
    return any(tok in c for tok in bad_tokens)

# Basic sequence-based counts
if "adt_event_types_seq_str" in df.columns:
    df["adt_event_count"] = df["adt_event_types_seq_str"].fillna("").apply(
        lambda x: len([t for t in str(x).split("|") if t != ""])
    )
if "proc_codes_seq_str" in df.columns:
    df["proc_code_count"] = df["proc_codes_seq_str"].fillna("").apply(
        lambda x: len([t for t in str(x).split("|") if t != ""])
    )

# Candidate numeric & categorical features
num_candidates = [
    "age", "med_order_nunique", "proc_count", "adt_event_count", "proc_code_count"
]
num_cols = [c for c in num_candidates if c in df.columns and not is_leaky(c)]

cat_candidates = [
    "PATIENT_SEX", "PATIENT_RACE_ETHNICITY", "encounter_setting", "encounter_medical_service"
]
cat_cols = [c for c in cat_candidates if c in df.columns and not is_leaky(c)]

if len(num_cols + cat_cols) == 0:
    raise ValueError("No usable features found after leakage filtering.")

In [6]:
# ------------------------------------------------------------
# 5) Train / test split
# ------------------------------------------------------------
X = df[num_cols + cat_cols].copy()
y = df["target_quantized"].astype(int)

# Handle missing values
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce")
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
for c in cat_cols:
    X[c] = X[c].astype("category").cat.add_categories("NA").fillna("NA")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y if y.nunique()==2 else None
)

In [7]:
# ------------------------------------------------------------
# 6) Model pipeline: OneHot + StandardScaler + Logistic Regression
# ------------------------------------------------------------
numeric_transformer = StandardScaler(with_mean=False)
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000, solver="lbfgs"))
])

clf.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(sparse_threshold=1.0,
                                   transformers=[('num',
                                                  StandardScaler(with_mean=False),
                                                  ['age', 'med_order_nunique',
                                                   'proc_count',
                                                   'adt_event_count',
                                                   'proc_code_count']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['PATIENT_SEX',
                                                   'PATIENT_RACE_ETHNICITY',
                                                   'encounter_setting',
                                                   'encounter_medical_service'])])),
                ('model', LogisticRegression(max_iter=2000))])

In [8]:
# ------------------------------------------------------------
# 7) Evaluation & exports
# ------------------------------------------------------------
y_pred  = clf.predict(X_test)
X_test_t = clf.named_steps["preprocess"].transform(X_test)
y_proba = clf.named_steps["model"].predict_proba(X_test_t)[:, 1]

report_dict = classification_report(y_test, y_pred, output_dict=True)
report_df   = pd.DataFrame(report_dict).T
cm          = confusion_matrix(y_test, y_pred)
roc         = roc_auc_score(y_test, y_proba) if y_test.nunique()==2 else np.nan
pr_auc      = average_precision_score(y_test, y_proba) if y_test.nunique()==2 else np.nan

# Feature coefficients (for interpretability)
ohe = clf.named_steps["preprocess"].named_transformers_.get("cat", None)
num_feature_names = num_cols
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist() if (ohe is not None and len(cat_cols) > 0) else []
feature_names = num_feature_names + cat_feature_names

coefs = clf.named_steps["model"].coef_.ravel()
feat_imp = pd.DataFrame({"feature": feature_names[:len(coefs)], "coef": coefs[:len(feature_names)]}) \
             .sort_values("coef", key=lambda s: s.abs(), ascending=False)

In [9]:
# ------------------------------------------------------------
# 8) Save results
# ------------------------------------------------------------
(SAVE_DIR / "metrics.txt").write_text(
    "=== Quantized MedSurg Stay Days Prediction (<=2 vs >2) — CLEAN (no leakage) ===\n"
    f"N train={len(X_train)}, N test={len(X_test)}\n"
    f"ROC-AUC: {roc:.4f}\nPR-AUC: {pr_auc:.4f}\n\n"
    "Classification Report:\n" + report_df.to_string() +
    "\n\nConfusion Matrix (rows=true, cols=pred):\n" + np.array2string(cm)
)

pred_out = X_test.copy()
pred_out["y_true"]  = y_test.values
pred_out["y_pred"]  = y_pred
pred_out["y_proba"] = y_proba
pred_out.to_csv(SAVE_DIR / "test_predictions_clean.csv", index=False)

feat_imp.to_csv(SAVE_DIR / "feature_coefficients_clean.csv", index=False)

print("Done. Clean baseline saved to:", SAVE_DIR.as_posix())
print("Numeric features:", num_cols)
print("Categorical features:", cat_cols)

Done. Clean baseline saved to: /Users/kolwu/Downloads
Numeric features: ['age', 'med_order_nunique', 'proc_count', 'adt_event_count', 'proc_code_count']
Categorical features: ['PATIENT_SEX', 'PATIENT_RACE_ETHNICITY', 'encounter_setting', 'encounter_medical_service']


In [10]:
pred_out.head(20)

,age,med_order_nunique,proc_count,adt_event_count,proc_code_count,PATIENT_SEX,PATIENT_RACE_ETHNICITY,encounter_setting,encounter_medical_service,y_true,y_pred,y_proba
23117,36.0,10.0,2.0,2,0,Female,"White, non-Hispanic",Outpatient,Laboratory,0,0,0.000689
53000,67.0,10.0,2.0,2,0,Male,"White, non-Hispanic",Other unspecified,Radiology,0,0,0.000003
53541,84.0,1.0,2.0,2,0,Male,"White, non-Hispanic",Outpatient,Nursing - medical / surgical,0,0,0.202848
36020,54.0,10.0,2.0,2,0,Male,"White, non-Hispanic",Other unspecified,Physical therapy,0,0,0.000148
36528,91.0,10.0,2.0,2,0,Female,"White, non-Hispanic",Other unspecified,Physical therapy,0,0,0.000223
1811,58.0,10.0,2.0,2,0,Male,"White, non-Hispanic",Other unspecified,Physical medicine and Rehab,0,0,0.001040
23685,37.0,1.0,1.0,2,1,Male,"White, non-Hispanic",Outpatient,Nephrology,0,0,0.000640
25244,52.0,10.0,2.0,2,0,Female,"White, non-Hispanic",Outpatient,Nutrition,0,0,0.006834
55980,84.0,10.0,2.0,2,0,Female,"White, non-Hispanic",Outpatient,Laboratory,0,0,0.000989
49223,6.0,8.0,3.0,8,3,Male,"White, non-Hispanic",Outpatient,Nursing - medical / surgical,1,1,0.837403


In [11]:
feat_imp

,feature,coef
4,proc_code_count,8.045301
2,proc_count,-6.389337
29,encounter_medical_service_Gastroenterology,5.205562
57,encounter_medical_service_Radiology,-4.922526
40,encounter_medical_service_Nephrology,-4.314040
...,...,...
49,encounter_medical_service_Pain medicine,-0.068393
22,encounter_medical_service_Anesthesiology,-0.017012
17,encounter_setting_NA,-0.006181
25,encounter_medical_service_Case management,-0.005907


In [12]:
print("\n=== Quantized MedSurg Stay Days Prediction (<=2 vs >2) ===")
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"ROC-AUC: {roc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")
print("\nConfusion Matrix (rows=true, cols=pred):")
print(cm)
print("\nClassification Report:")
print(report_df.round(3))


=== Quantized MedSurg Stay Days Prediction (<=2 vs >2) ===
Train size: 46299, Test size: 15434
ROC-AUC: 0.9882
PR-AUC:  0.9588

Confusion Matrix (rows=true, cols=pred):
[[11171   418]
 [  216  3629]]

Classification Report:
              precision  recall  f1-score    support
0                 0.981   0.964     0.972  11589.000
1                 0.897   0.944     0.920   3845.000
accuracy          0.959   0.959     0.959      0.959
macro avg         0.939   0.954     0.946  15434.000
weighted avg      0.960   0.959     0.959  15434.000
